# BIOM 200: Our nasal microbiomes

Instructors: Rob Knight (robknight@ucsd.edu), Lucas Patel (lpatel@health.ucsd.edu)

On Thursday, October 15, we swabbed the left and right nostril of each participant, students and instructors alike. The swabs were extracted, amplified for the 16S rRNA V4 region, sequenced the same day, and processed in Qiita (study 16596). This notebook asks three questions of those data:

1. **Left vs. right:** do the two nostrils of the same person host different communities?
2. **Self vs. others:** is your left nostril more like your right nostril than like anyone else's?
3. **Students vs. instructors:** do the two groups differ?

Before running anything, write down what you expect for each question and why. It reuses the functions from the first notebook (`scripts/biom200.py`); if a step is unfamiliar, the matching section there explains it.

## The nasal microbiome

The anterior nares are dominated by a small number of genera, typically *Corynebacterium*, *Cutibacterium*, *Staphylococcus*, *Dolosigranulum*, and *Moraxella*, and *Staphylococcus aureus* is carried by a substantial fraction of healthy adults. Body site explains more of the variation in the human microbiome than anything else, but within a site, individuals differ consistently from one another. Nasal swabs are also low in biomass, so contamination from reagents is a real risk.

**Read more:** [Bomar et al. 2018, *Curr Opin Microbiol*](https://doi.org/10.1016/j.mib.2017.10.023) · [Human Microbiome Project Consortium 2012, *Nature*](https://doi.org/10.1038/nature11234) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486)

In [ ]:
%matplotlib inline
from scripts.biom200 import *

## Load the class data

The files are downloaded from Qiita study 16596 into `data/class/`. **EDIT** `INSTRUCTORS` with the participant IDs (`host_subject_id`) of the instructors; everyone else is labelled a student.

In [ ]:
# EDIT: class study files (Qiita study 16596)
TABLE = "data/class/16596_feature-table.biom"
METADATA = "data/class/16596_sample_information.tsv"
TAXONOMY = "data/class/16596_taxonomy.tsv"
TREE = "data/class/16596_tree.nwk"

# EDIT: participant IDs of the instructors
INSTRUCTORS = []

nasal = load_dataset("nasal", TABLE, METADATA, TAXONOMY, TREE, group="side", subject="host_subject_id",
                     control_col="sample_type", control_values=["control blank"])
add_metadata_column(nasal, "role", lambda m: np.where(m["host_subject_id"].isin(INSTRUCTORS), "instructor", "student"))

## Controls: what do the blanks tell us?

Each extraction plate carried blanks: wells with no sample, processed exactly like the swabs. Reads in a blank come from reagents, from the lab, or from neighbouring wells. Compare the blanks with the swabs: how many reads do they have, and which taxa? A blank with as many reads as a real sample, or containing the dominant taxa of a neighbouring sample, points to well-to-well contamination or a swapped tube rather than reagent background.

**Read more:** [Salter et al. 2014, *BMC Biol*](https://doi.org/10.1186/s12915-014-0087-z) · [Minich et al. 2019, *mSystems*](https://doi.org/10.1128/mSystems.00186-19)

In [ ]:
explore_controls(nasal)

In [ ]:
control_report(nasal).head(10)

## Composition

Which genera dominate our noses, and do they match the expected nasal taxa? Group by `side` and by `role`, and tick **Group means** to compare the averages.

**Read more:** [Bomar et al. 2018, *Curr Opin Microbiol*](https://doi.org/10.1016/j.mib.2017.10.023)

In [ ]:
add_composition(nasal)
explore_composition(nasal)

## Rarefaction

Read depth varies over several orders of magnitude between our samples. Choose a depth where most curves have flattened, and note which samples fall below it. Losing one nostril of a person also removes that person from the paired comparisons below.

**Read more:** [Weiss et al. 2017, *Microbiome*](https://doi.org/10.1186/s40168-017-0237-y)

In [ ]:
add_curves(nasal)
explore_rarefaction(nasal)

In [ ]:
add_diversity(nasal)

## Question 1: left vs. right

Each person contributes one left and one right sample, so the two sides are **paired**. A paired test compares each person's left with their own right, which removes the large differences between people from the comparison. Each line below joins one person's two nostrils.

In [ ]:
explore_paired_alpha(nasal, group="side", pair_by="host_subject_id")

The same question for the whole community: in the PCoA, grey lines join the two nostrils of each person. Short lines mean the two sides are similar. Note that PERMANOVA by `side` treats all samples as independent; with paired samples, the comparison in Question 2 is more informative.

In [ ]:
explore_pcoa(nasal, connect="host_subject_id")

## Question 2: are you more like yourself than like anyone else?

Compare the distance between your own two nostrils with the distances between nostrils of different people. If individuals have consistent nasal communities, same-person distances should be smaller.

**Read more:** [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Lozupone et al. 2011, *ISME J*](https://doi.org/10.1038/ismej.2010.133)

In [ ]:
explore_within_between(nasal, by="host_subject_id")

In [ ]:
explore_distance_matrix(nasal)

PERMANOVA for side, participant, and role. Participant usually explains far more than side.

**Read more:** [Anderson 2001, *Austral Ecol*](https://doi.org/10.1111/j.1442-9993.2001.01070.pp.x)

In [ ]:
permanova_table(nasal, ["side", "host_subject_id", "role"])

## Question 3: students vs. instructors

Use **Group by → role** in the plots above, or the cells below. Each person contributes two samples, so the samples are not independent: with *n* people, there are only *n* independent observations, not 2*n*. Be cautious with any p-value here, and ask what else differs between the two groups besides being an instructor.

In [ ]:
explore_alpha(nasal)

## Differential abundance

First, left vs. right adjusted for participant, so that each person is compared with themselves. Then students vs. instructors. With this few people, treat every result as a hypothesis.

**Read more:** [Lin & Peddada 2020, *Nat Commun*](https://doi.org/10.1038/s41467-020-17041-7) · [Nearing et al. 2022, *Nat Commun*](https://doi.org/10.1038/s41467-022-28034-z)

In [ ]:
add_differential_abundance(nasal, group="side", covariates=["host_subject_id"], min_prevalence=0.25)
explore_forest(nasal, group="side")

In [ ]:
add_differential_abundance(nasal, group="role", min_prevalence=0.25)
explore_forest(nasal, group="role")

## Reflection

1. What did you predict for each question, and what did you find?
2. Which result would you defend in a paper, and which would you not? Why?
3. What do the blanks and the low-read samples tell you about the limits of these data?
4. If you could redesign the study, what would you change: number of people, swabs per nostril, controls, metadata?

**Read more:** [Knight et al. 2018, *Nat Rev Microbiol*](https://doi.org/10.1038/s41579-018-0029-9)